# DPO Smoke Test — Confirm the Plumbing Works

**Purpose:** before running real DPO training, confirm every piece of the pipeline works end-to-end on the smallest possible scale. This notebook deliberately does **no real training** — it runs *one* training step with *two* examples and stops.

**Why this matters:** real DPO training takes 1-2 hours on a T4. If something is misconfigured (wrong library version, model doesn't fit in memory, tokenizer mismatch, dataset format issue), we'd rather discover it in 5 minutes here than waste 90 minutes finding out from a real training run.

**What this notebook verifies:**
1. GPU is available
2. The base model loads into GPU memory
3. Inference works (the model can generate text)
4. The FormatBench dataset loads correctly
5. The DPOTrainer can be instantiated
6. One training step completes without error

**Expected runtime:** ~5-10 minutes on a Kaggle T4.

**Kaggle setup:**
1. Create new Kaggle Notebook
2. Settings → Accelerator → **GPU T4 x2** (or any GPU)
3. Settings → Internet → **On**
4. Paste this notebook in and run all cells

---
*Part of the [Prosify project](https://github.com/[your-username]/prosify).*

## 1. Install dependencies

We need recent versions of `transformers`, `trl`, `datasets`, and `accelerate`. Kaggle's pre-installed versions may be outdated.

In [ ]:
!pip install -q -U transformers trl datasets accelerate

import transformers, trl, datasets, accelerate, torch
print(f"transformers: {transformers.__version__}")
print(f"trl:          {trl.__version__}")
print(f"datasets:     {datasets.__version__}")
print(f"accelerate:   {accelerate.__version__}")
print(f"torch:        {torch.__version__}")

## 2. Configuration

Change `HF_USERNAME` to your HuggingFace username before running.

In [ ]:
# ===== EDIT THIS =====
HF_USERNAME = "your-hf-username"  # <-- your HuggingFace username
# =====================

DATASET_REPO = f"{HF_USERNAME}/formatbench"
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  # small, fits on T4, instruction-tuned

## 3. Confirm GPU is available

If `CUDA available: False`, the rest of the notebook will fail. Go to **Settings → Accelerator** and pick a GPU runtime.

In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU memory:     {mem_gb:.1f} GB")
else:
    raise RuntimeError("No GPU detected. Enable GPU in Kaggle Notebook settings.")

## 4. Load the base model and tokenizer

We load Qwen 2.5 1.5B Instruct — a small, instruction-tuned model that fits comfortably in T4 memory (~3 GB for the model itself). We use `float16` for memory efficiency.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Loading {BASE_MODEL}...")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
)

# Some tokenizers don't have a pad token by default; DPO needs one
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

n_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f"\nModel loaded.")
print(f"  Parameters:     {n_params:.2f}B")
print(f"  Device:         {next(model.parameters()).device}")
print(f"  Dtype:          {next(model.parameters()).dtype}")
print(f"  GPU mem in use: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 5. Test inference

Generate one response to a sample prompt. This confirms the model loaded correctly and can produce output. The response itself doesn't matter — we just need to see *something* coherent come out.

In [ ]:
sample_prompt = "Write a one-sentence email letting my manager know I'll be late tomorrow."

# Format with chat template (Qwen is instruction-tuned)
messages = [{"role": "user", "content": sample_prompt}]
formatted = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(formatted, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )

response = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(f"PROMPT: {sample_prompt}\n")
print(f"RESPONSE: {response}")

## 6. Load the FormatBench dataset (just 2 examples for the smoke test)

For the smoke test we only need a tiny slice. The real training notebook will use the full train split.

In [ ]:
from datasets import load_dataset

full_train = load_dataset(DATASET_REPO, split="train")
tiny_train = full_train.select(range(2))  # just 2 examples

print(f"Full training set: {len(full_train)} examples")
print(f"Tiny smoke test set: {len(tiny_train)} examples")
print(f"\nFirst example keys: {list(tiny_train[0].keys())}")
print(f"\nFirst prompt: {tiny_train[0]['prompt'][:100]}...")

## 7. Format prompts with the chat template

Qwen expects prompts wrapped in its chat format. We apply the chat template to the `prompt` field so DPOTrainer sees the model's expected input format. The `chosen` and `rejected` fields stay as plain strings (they're the assistant's response in each case).

In [ ]:
def format_with_chat_template(example):
    formatted_prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": example["prompt"]}],
        tokenize=False,
        add_generation_prompt=True,
    )
    return {
        "prompt": formatted_prompt,
        "chosen": example["chosen"],
        "rejected": example["rejected"],
    }

tiny_train_formatted = tiny_train.map(
    format_with_chat_template,
    remove_columns=[c for c in tiny_train.column_names if c not in ["prompt", "chosen", "rejected"]],
)

print("Formatted prompt example (first 200 chars):")
print(tiny_train_formatted[0]["prompt"][:200])
print("...")
print(f"\nColumns: {tiny_train_formatted.column_names}")

## 8. Set up the DPOTrainer (minimal config)

This is the configuration that runs DPO training. For the smoke test, every setting is at its minimal value:
- Batch size 1
- Max 1 training step
- No checkpointing
- Small learning rate

The point is to confirm the trainer can be instantiated and runs one step — not to learn anything useful.

In [ ]:
from trl import DPOTrainer, DPOConfig

smoke_config = DPOConfig(
    output_dir="./dpo_smoke_test_out",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    max_steps=1,                # <-- ONE step only
    learning_rate=5e-6,
    beta=0.1,                   # standard DPO KL-leash strength
    max_length=1024,            # caps the total input length
    max_prompt_length=512,      # caps just the prompt portion
    logging_steps=1,
    save_strategy="no",         # don't save checkpoints in smoke test
    report_to="none",           # don't log to wandb / tensorboard
    remove_unused_columns=False,
    bf16=False,                 # T4 doesn't fully support bf16
    fp16=True,
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,             # if None, TRL silently creates a frozen copy
    args=smoke_config,
    train_dataset=tiny_train_formatted,
    processing_class=tokenizer,
)

print("✓ DPOTrainer created successfully")
print(f"  Train examples: {len(trainer.train_dataset)}")
print(f"  Max steps:      {smoke_config.max_steps}")

## 9. Run ONE training step

If this cell runs without error, the pipeline is working end-to-end. You're ready for real DPO training in the next notebook.

In [ ]:
print("Running one DPO training step...\n")
trainer.train()
print("\n✓ Training step completed without error.")

## 10. Smoke test verdict

If you got here without errors, all of these are confirmed working:

✅ GPU is available and has enough memory  
✅ Qwen 2.5 1.5B loads correctly in fp16  
✅ The model can generate text (inference works)  
✅ FormatBench loads from HuggingFace  
✅ The chat template applies correctly to prompts  
✅ DPOTrainer accepts the formatted dataset  
✅ One training step runs end-to-end  

**The environment is ready for real DPO training.**

---

**Next step:** in the next notebook (`dpo_02_train.ipynb`), we'll run actual DPO training — full training split, 1-2 epochs, multiple hyperparameter combinations, save the best checkpoint.

**If you got an error,** it's almost certainly in one of these categories:
- Library version mismatch → check the printed versions in cell 1, may need pinned versions
- Out of memory → reduce `max_length` to 512, or switch to a smaller model
- Authentication → some models require HuggingFace login (Qwen 2.5 1.5B-Instruct doesn't, but worth checking)
- Chat template error → verify the model name is spelled correctly

Whatever the error, paste it back and we'll diagnose before moving to real training.